# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** Refresh / Content Opportunity Scoring. This notebook uses the same warehouse definition as ML-04: March 2026 data is available at decision time; April 2026 is used only as a retrospective evaluation outcome. The score, rule, reason, action, and exported queue do not use April data or the decline label.

## 1. Check the signals before writing the rule

The two bucket tables compare March-only signals with the April decline proxy. Each table reports n. These outcomes are audit/evaluation targets only; they are not inputs to the scoring rule.

- **Volume: MIXED.** March impressions are a real opportunity/visibility signal, but the April decline rate is not monotonic across the four buckets. Higher visibility alone is not a reliable risk rule.
- **CTR within position bands: CONFIRMED (association).** In both position bands, pages with March CTR below 0.5% have higher April decline rates than pages at or above 0.5%. This supports a human snippet/page review flag, not a causal claim or proof that a rewrite will recover clicks.

## One transparent rule

Queue pages for a **snippet and landing-page review** when March impressions are at least 500, average position is in the top 20, and CTR is below 0.5%. Rank those candidates by `log1p(March impressions)`, so higher observed search exposure is reviewed first. Each queued item receives exactly one reason code (`high_visibility_low_ctr`) and one action label (`review_snippet_and_page`). The thresholds are a readable baseline, not learned weights.

**Leakage boundary:** April impressions create the retrospective decline target only. They, the target, and all derived trend fields are excluded from the score and CSV.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
while not (repo_root / '.env').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / '.env').is_file():
    raise FileNotFoundError('Repository-root .env with HF_TOKEN is required; the token is never displayed.')
hf_token = None
for line in (repo_root / '.env').read_text(encoding='utf-8').splitlines():
    if '=' in line and not line.lstrip().startswith('#'):
        key, value = line.split('=', 1)
        if key.strip() == 'HF_TOKEN':
            hf_token = value.strip().strip(chr(34)).strip(chr(39))
            break
if not hf_token or not hf_token.startswith('hf_'):
    raise ValueError('HF_TOKEN is missing or malformed in .env; token value was not displayed.')
con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute('CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)', [hf_token])
del hf_token
warehouse = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
march_source = 'read_parquet(' + repr(warehouse + '/month=2026-03/*.parquet') + ')'
april_source = 'read_parquet(' + repr(warehouse + '/month=2026-04/*.parquet') + ')'
print('Warehouse connected using the ignored .env; no credential was printed.')

Warehouse connected using the ignored .env; no credential was printed.


In [2]:
feature_sql = f'''
WITH march AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_mar,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_mar,
           SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS clicks_mar,
           SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS position_sum_mar
    FROM {march_source}
    GROUP BY client_hash_id, content_hash_id
), april AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_apr,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_apr
    FROM {april_source}
    GROUP BY client_hash_id, content_hash_id
)
SELECT march.client_hash_id, march.content_hash_id,
       march.gsc_days_mar, march.impressions_mar, march.clicks_mar,
       march.position_sum_mar / NULLIF(march.impressions_mar, 0) AS avg_position_mar,
       100.0 * march.clicks_mar / NULLIF(march.impressions_mar, 0) AS ctr_mar,
       CAST(april.impressions_apr <= 0.8 * march.impressions_mar AS INTEGER) AS april_decline_label
FROM march INNER JOIN april USING (client_hash_id, content_hash_id)
WHERE march.gsc_days_mar >= 20 AND april.gsc_days_apr >= 20
  AND march.impressions_mar > 0
'''
items = con.execute(feature_sql).df()
LABEL = 'april_decline_label'
ID_COLUMNS = ['client_hash_id', 'content_hash_id']
assert len(items) == 95633
assert items[ID_COLUMNS].duplicated().sum() == 0
assert items[LABEL].isin([0, 1]).all()
print(f'Eligible item count: {len(items):,}; April decline base rate for evaluation only: {items[LABEL].mean():.3f}')
print('Scoring uses only March impressions, average position, and CTR. April outcome is held aside for evaluation.')

Eligible item count: 95,633; April decline base rate for evaluation only: 0.495
Scoring uses only March impressions, average position, and CTR. April outcome is held aside for evaluation.


### March feature buckets and April evaluation outcomes

The first table tests visibility volume. The second tests low CTR within position bands so position is not mistaken for a snippet problem. Counts below 50 would be treated as too small for a verdict.

In [3]:
signal_items = items.copy()
signal_items['volume_bucket'] = pd.cut(
    signal_items['impressions_mar'],
    bins=[0, 100, 500, 2000, np.inf],
    labels=['<100', '100-499', '500-1,999', '2,000+'],
    right=False,
    include_lowest=True,
)
volume_table = (
    signal_items.groupby('volume_bucket', observed=False)
    .agg(n=(LABEL, 'size'), april_decline_rate=(LABEL, 'mean'))
    .reset_index()
)
print('Signal 1 — monthly search volume (March impressions):')
display(volume_table.assign(april_decline_rate=volume_table['april_decline_rate'].round(3)))

signal_items['position_band'] = np.select(
    [signal_items['avg_position_mar'] <= 10, signal_items['avg_position_mar'] <= 20],
    ['position 1-10', 'position 11-20'],
    default='outside position 1-20',
)
signal_items['ctr_bucket'] = np.where(signal_items['ctr_mar'] < 0.5, 'CTR <0.5%', 'CTR >=0.5%')
signal_items['ctr_position_bucket'] = np.where(
    signal_items['position_band'].eq('outside position 1-20'),
    signal_items['position_band'],
    signal_items['position_band'] + ' / ' + signal_items['ctr_bucket'],
)
ctr_position_table = (
    signal_items.groupby('ctr_position_bucket', observed=True)
    .agg(n=(LABEL, 'size'), april_decline_rate=(LABEL, 'mean'))
    .reset_index()
    .sort_values('ctr_position_bucket')
)
print('Signal 2 — CTR buckets within March position bands:')
display(ctr_position_table.assign(april_decline_rate=ctr_position_table['april_decline_rate'].round(3)))
assert volume_table['n'].min() >= 50
assert ctr_position_table['n'].min() >= 50

Signal 1 — monthly search volume (March impressions):


,volume_bucket,n,april_decline_rate
0,<100,6692,0.254
1,100-499,30137,0.518
2,"500-1,999",29575,0.544
3,"2,000+",29229,0.477


Signal 2 — CTR buckets within March position bands:

,ctr_position_bucket,n,april_decline_rate
0,outside position 1-20,23906,0.464
1,position 1-10 / CTR <0.5%,42398,0.553
2,position 1-10 / CTR >=0.5%,10378,0.319
3,position 11-20 / CTR <0.5%,16008,0.522
4,position 11-20 / CTR >=0.5%,2943,0.373


## 2. Encode the one-rule ranked queue

The eligibility rule and score use March fields only. April's label is not added to the queue. The export includes pseudonymous IDs for downstream joins, but does not display them in the notebook review.

In [4]:
target_by_id = items.set_index(ID_COLUMNS)[LABEL]
condition = (
    (items['impressions_mar'] >= 500)
    & (items['avg_position_mar'] > 0)
    & (items['avg_position_mar'] <= 20)
    & (items['ctr_mar'] < 0.5)
)
score_columns = ID_COLUMNS + ['gsc_days_mar', 'impressions_mar', 'clicks_mar', 'avg_position_mar', 'ctr_mar']
eligible = items.loc[condition, score_columns].copy()
eligible['baseline_score'] = np.log1p(eligible['impressions_mar'])
eligible['reason_code'] = 'high_visibility_low_ctr'
eligible['action_label'] = 'review_snippet_and_page'
queue_columns = ID_COLUMNS + [
    'baseline_score', 'reason_code', 'action_label', 'impressions_mar',
    'clicks_mar', 'avg_position_mar', 'ctr_mar', 'gsc_days_mar',
]
queue = eligible.sort_values(
    ['baseline_score', 'clicks_mar', 'avg_position_mar'],
    ascending=[False, False, True],
    kind='mergesort',
).reset_index(drop=True)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))
assert queue['reason_code'].nunique() == 1
assert queue['action_label'].nunique() == 1
assert not set([LABEL, 'impressions_apr', 'trend_pct', 'trend_direction']) & set(queue.columns)
output_dir = repo_root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / 'baseline_action_score.csv'
queue[['rank'] + queue_columns].to_csv(queue_path, index=False)
print(f'Queue written to {queue_path.relative_to(repo_root)} ({len(queue):,} candidates).')
print('Single reason code:', queue['reason_code'].iloc[0], '| single action:', queue['action_label'].iloc[0])
queue_keys = pd.MultiIndex.from_frame(queue[ID_COLUMNS])
queue_labels = target_by_id.reindex(queue_keys).to_numpy(dtype=int)
base_rate = items[LABEL].mean()
def precision_at_k(y_true, k):
    return float(np.mean(y_true[:min(k, len(y_true))]))
print(f'Eligible population April decline base rate (evaluation only): {base_rate:.3f}')
for k in (10, 20, 50):
    print(f'Rule Precision@{k}: {precision_at_k(queue_labels, k):.3f} ({min(k, len(queue)):,} reviewed)')
print('The evaluation label did not participate in selecting or sorting the queue.')

Queue written to work\outputs\baseline_action_score.csv (39,545 candidates).
Single reason code: high_visibility_low_ctr | single action: review_snippet_and_page
Eligible population April decline base rate (evaluation only): 0.495
Rule Precision@10: 0.400 (10 reviewed)
Rule Precision@20: 0.350 (20 reviewed)
Rule Precision@50: 0.400 (50 reviewed)
The evaluation label did not participate in selecting or sorting the queue.


## 3. Skeptical top-10 review

Every line shows the action, why that item ranked here, and a plausible reason the recommendation could be wrong. The review displays feature metrics, not page/client identifiers.

In [5]:
top10 = queue.head(10).copy()
def review_risk(row):
    if row['avg_position_mar'] <= 10:
        return 'Low CTR may come from SERP features, query intent, or a brand-heavy query mix rather than stale page content.'
    return 'Position beyond the top 10 may be the main CTR constraint; changing the snippet alone may not improve visibility.'

top10['why_here'] = top10.apply(
    lambda row: f"{row['impressions_mar']:,.0f} March impressions, position {row['avg_position_mar']:.2f}, CTR {row['ctr_mar']:.3f}% (<0.5%).",
    axis=1,
)
top10['what_would_make_it_wrong'] = top10.apply(review_risk, axis=1)
top10['review_line'] = top10.apply(
    lambda row: f"Rank {row['rank']}: {row['action_label']} because {row['why_here']} Could be wrong if {row['what_would_make_it_wrong']}",
    axis=1,
)
for line in top10['review_line']:
    print(line)
assert len(top10) == 10

Rank 1: review_snippet_and_page because 244,931 March impressions, position 15.17, CTR 0.273% (<0.5%). Could be wrong if Position beyond the top 10 may be the main CTR constraint; changing the snippet alone may not improve visibility.
Rank 2: review_snippet_and_page because 221,310 March impressions, position 2.51, CTR 0.325% (<0.5%). Could be wrong if Low CTR may come from SERP features, query intent, or a brand-heavy query mix rather than stale page content.
Rank 3: review_snippet_and_page because 212,404 March impressions, position 0.67, CTR 0.011% (<0.5%). Could be wrong if Low CTR may come from SERP features, query intent, or a brand-heavy query mix rather than stale page content.
Rank 4: review_snippet_and_page because 205,867 March impressions, position 3.30, CTR 0.419% (<0.5%). Could be wrong if Low CTR may come from SERP features, query intent, or a brand-heavy query mix rather than stale page content.
Rank 5: review_snippet_and_page because 203,497 March impressions, position

## 4. Weak picks and retrospective check

A weak pick is a queued item whose April decline label is 0. This label is used only after ranking to inspect errors and calculate precision; it is not a feature or a CSV field.

In [6]:
top10_diagnostic = top10[['rank'] + ID_COLUMNS].copy()
top10_diagnostic['april_decline_label'] = target_by_id.reindex(
    pd.MultiIndex.from_frame(top10_diagnostic[ID_COLUMNS])
).to_numpy(dtype=int)
weak_picks = top10_diagnostic.loc[top10_diagnostic['april_decline_label'].eq(0), ['rank', 'april_decline_label']]
print(f'Weak picks in top ten: {len(weak_picks)} of 10.')
if len(weak_picks):
    print('Anonymous queue ranks that did not meet the April decline proxy:')
    display(weak_picks.reset_index(drop=True))
else:
    print('No top-ten weak picks in this run; do not infer that the rule is reliable from ten cases.')
print('Important: a non-decline label does not prove a page did not need editorial work, and a decline does not prove a refresh would help.')
con.close()

Weak picks in top ten: 6 of 10.
Anonymous queue ranks that did not meet the April decline proxy:


,rank,april_decline_label
0,2,0
1,4,0
2,5,0
3,7,0
4,8,0
5,9,0


Important: a non-decline label does not prove a page did not need editorial work, and a decline does not prove a refresh would help.


## 5. Self-check

- Two signal bucket tables show n and retrospective April decline rates; verdicts are stated above.
- The queue uses one transparent rule, one reason code, and one action label.
- Scoring uses March metrics only; April and its label are evaluation-only.
- The CSV is generated under `work/outputs/` and is intentionally not committed; the executed notebook contains the reproducible rule and review.
- The top ten are reviewed without printing page or client IDs; weak picks are called out using anonymous ranks.
- Neither score nor recommendation implies that a refresh causes recovery.